# Flow Matching on $\mathrm{SO}(3)$
In this notebook, we perform flow matching on the rotation group $\mathrm{SO}(3)$, corresponding to Figure 2 in ["Flow Matching on Lie Groups"](https://arxiv.org/abs/2504.00494).

In [ ]:
import torch
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
import numpy as np
from lieflow.groups import SO3
from lieflow.models import (
    get_model_FM,
    LogarithmicDistance,
    DataGenerator,
)
from lieflow.metrics import compare
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.cm import ScalarMappable
from tqdm.notebook import tqdm
from functools import partial
import json
with open("plot.json", "r") as f:
    plot_config = json.load(f)
latex_line_width = plot_config.pop("latex_line_width")
plt.rcParams.update(plot_config)

device = "cpu"
SEED = 0
torch.manual_seed(SEED);

In [ ]:
so3 = SO3(device=device)

In [ ]:
generate_videos = True

# Distributions

In [ ]:
def generate_line(N, d=0., w=torch.pi/4, horizontal=True, ε=0.05):
    if horizontal:
        d_ε = d + torch.randn(N, device=device) * ε
        cos_y = torch.cos(d_ε)
        sin_y = torch.sin(d_ε)
        R_y = torch.zeros(N, 3, 3, device=device)
        R_y[..., 0, 0] = cos_y
        R_y[..., 0, 2] = -sin_y
        R_y[..., 1, 1] = 1.
        R_y[..., 2, 0] = sin_y
        R_y[..., 2, 2] = cos_y
        
        θs_z = 2 * w * (torch.rand(N, device=device) - 0.5)
        cos_z = torch.cos(θs_z)
        sin_z = torch.sin(θs_z)
        R_z = torch.zeros(N, 3, 3, device=device)
        R_z[..., 0, 0] = cos_z
        R_z[..., 0, 1] = -sin_z
        R_z[..., 1, 0] = sin_z
        R_z[..., 1, 1] = cos_z
        R_z[..., 2, 2] = 1.

        Rs = R_z @ R_y
    else:
        turn_ε = -torch.pi/2 + torch.randn(N, device=device) * ε
        cos_turn = torch.cos(turn_ε)
        sin_turn = torch.sin(turn_ε)
        R_turn = torch.zeros(N, 3, 3, device=device)
        R_turn[..., 0, 0] = 1.
        R_turn[..., 1, 1] = cos_turn
        R_turn[..., 1, 2] = -sin_turn
        R_turn[..., 2, 1] = sin_turn
        R_turn[..., 2, 2] = cos_turn
        
        d_ε = d + torch.randn(N, device=device) * ε
        cos_z = torch.cos(d_ε)
        sin_z = torch.sin(d_ε)
        R_z = torch.zeros(N, 3, 3, device=device)
        R_z[..., 0, 0] = cos_z
        R_z[..., 0, 1] = -sin_z
        R_z[..., 1, 0] = sin_z
        R_z[..., 1, 1] = cos_z
        R_z[..., 2, 2] = 1.

        θs_y = 2 * w * (torch.rand(N, device=device) - 0.5)
        cos_y = torch.cos(θs_y)
        sin_y = torch.sin(θs_y)
        R_y = torch.zeros(N, 3, 3, device=device)
        R_y[..., 0, 0] = cos_y
        R_y[..., 0, 2] = -sin_y
        R_y[..., 1, 1] = 1.
        R_y[..., 2, 0] = sin_y
        R_y[..., 2, 2] = cos_y

        Rs = R_z @ R_y @ R_turn
    return Rs

def generate_circle(N, d=0., w=torch.pi/4, ε=0.05, gap=0.):
    w_ε = w + torch.randn(N, device=device) * ε
    cos_y = torch.cos(w_ε)
    sin_y = torch.sin(w_ε)
    R_y = torch.zeros(N, 3, 3, device=device)
    R_y[..., 0, 0] = cos_y
    R_y[..., 0, 2] = -sin_y
    R_y[..., 1, 1] = 1.
    R_y[..., 2, 0] = sin_y
    R_y[..., 2, 2] = cos_y

    θs_x = (torch.rand(N, device=device) * (2 - gap) - (1.5 - gap)) * torch.pi
    cos_x = torch.cos(θs_x)
    sin_x = torch.sin(θs_x)
    R_x = torch.zeros(N, 3, 3, device=device)
    R_x[..., 0, 0] = 1.
    R_x[..., 1, 1] = cos_x
    R_x[..., 1, 2] = -sin_x
    R_x[..., 2, 1] = sin_x
    R_x[..., 2, 2] = cos_x
    
    d_ε = d + torch.randn(N, device=device) * ε
    cos_z = torch.cos(d_ε)
    sin_z = torch.sin(d_ε)
    R_z = torch.zeros(N, 3, 3, device=device)
    R_z[..., 0, 0] = cos_z
    R_z[..., 0, 1] = -sin_z
    R_z[..., 1, 0] = sin_z
    R_z[..., 1, 1] = cos_z
    R_z[..., 2, 2] = 1.
    
    return R_z @ R_x @ R_y

q_ref = torch.tensor([
    [1., 0., 0.],
    [0., 0., 1.]
    ]).T

@torch.no_grad()
def plot_R(ax, R, **kwargs):
    R = R.to("cpu")
    q = R @ q_ref
    r = 1.01
    return ax.quiver(
        r*q[..., 0, 0], r*q[..., 1, 0], r*q[..., 2, 0],
        q[..., 0, 1], q[:32, 1, 1], q[..., 2, 1],
        length=0.1, linewidths=0.1, **kwargs
    )

In [ ]:
tests = ("horizontal_line_to_vertical_line", "vertical_line_to_circle")

In [ ]:
def data_generator(test, ε=0.04):
    match test:
        case "horizontal_line_to_vertical_line":
            generate_R_0 = partial(generate_line, d=-torch.pi/6, w=torch.pi/12, horizontal=True, ε=ε)
            generate_R_1 = partial(generate_line, d=torch.pi/6, w=torch.pi/12, horizontal=False, ε=ε)
        case "vertical_line_to_vertical_line":
            generate_R_0 = partial(generate_line, d=-torch.pi/6, w=torch.pi/12, horizontal=False, ε=ε)
            generate_R_1 = partial(generate_line, d=torch.pi/6, w=torch.pi/12, horizontal=False, ε=ε)
        case "vertical_line_to_circle":
            generate_R_0 = partial(generate_line, d=-torch.pi/6, w=torch.pi/12, horizontal=False, ε=ε)
            generate_R_1 = partial(generate_circle, d=torch.pi/6, w=torch.pi/12, gap=0., ε=ε)
        case "circle_to_circle":
            generate_R_0 = partial(generate_circle, w=torch.pi/20, ε=ε)
            generate_R_1 = partial(generate_circle, w=torch.pi/6, ε=ε)
    return generate_R_0, generate_R_1

# Models

## Training

In [ ]:
with open("config.json", "r") as f:
    config_all = json.load(f)
config_space = config_all["SO3"]
# Default parameters, overridden by those of the space, which are in turn overridden by those of the test case.
config_shared = config_all["defaults"] | {key: value for key, value in config_space.items() if not isinstance(value, dict)}
config = {test: config_shared | value for test, value in config_space.items() if isinstance(value, dict)}

In [ ]:
N_EVAL = 2**8

eval_data = {}
for test in tests:
    generate_R_0, generate_R_1 = data_generator(test, ε=config[test]["epsilon"])
    R_0s_eval, R_1s_eval, R_1s_ref = generate_R_0(N_EVAL), generate_R_1(N_EVAL), generate_R_1(N_EVAL)
    eval_data[test] = (R_0s_eval, R_1s_eval, compare(so3, R_1s_ref, R_1s_eval))

In [ ]:
def train_model(generate_R_0, generate_R_1, config, R_0s_eval, R_1s_eval, floor):
    train_loader = DataGenerator(generate_R_0, generate_R_1, config["batch_size"], config["batches_per_epoch"])

    model_FM = get_model_FM(
        so3, H=config["width"], L=config["depth"], activation=getattr(torch.nn, config["activation"])
    ).to(device)
    print("Number of parameters: ", model_FM.parameter_count)
    
    optimizer_FM = torch.optim.Adam(model_FM.parameters(), config["learning_rate"], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_FM, T_max=config["epochs"] * config["batches_per_epoch"], eta_min=config["learning_rate_final"]
    )
    ema = AveragedModel(model_FM, multi_avg_fn=get_ema_multi_avg_fn(config["ema_decay"]))
    loss = LogarithmicDistance(torch.tensor(config["loss_weights"], device=device))

    losses_FM = np.zeros(config["epochs"])
    metrics_FM = {key: np.zeros(config["epochs"]) for key in floor}
    for i in tqdm(range(config["epochs"])):
        losses_FM[i] = model_FM.train_network(device, train_loader, optimizer_FM, scheduler, ema, loss, show_progress=False)
        for key, value in ema.module.evaluate(R_0s_eval, R_1s_eval, N_steps=config["N_steps"]).items():
            metrics_FM[key][i] = value

    epochs = np.arange(1, config["epochs"] + 1)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
    ax[0].plot(epochs, losses_FM)
    ax[0].set_title("Batch Loss Flow Matching")
    ax[0].set_yscale("log")
    labels = {"W_1": "$W_1$", "precision": "precision", "coverage": "coverage"}
    for (key, label), color in zip(labels.items(), ("tab:blue", "tab:orange", "tab:green")):
        ax[1].plot(epochs, metrics_FM[key], color=color, label=label)
        ax[1].axhline(floor[key], color=color, linestyle="--", linewidth=0.8)
    ax[1].set_title("Distances (dashed: floor)")
    ax[1].set_yscale("log")
    ax[1].legend()
    ax[2].plot(epochs, metrics_FM["MMD^2"], color="tab:red")
    ax[2].axhline(floor["MMD^2"], color="tab:red", linestyle="--", linewidth=0.8)
    ax[2].set_title("MMD$^2$ (dashed: floor)")
    for a in ax:
        a.set_xscale("log")
        a.set_xlabel("Epoch")

    return ema.module, losses_FM, metrics_FM

In [ ]:
theta = torch.linspace(0, torch.pi, 50)
phi = torch.linspace(0, 2 * torch.pi, 100)
theta, phi = torch.meshgrid(theta, phi)

x = torch.sin(theta) * torch.cos(phi)
y = torch.sin(theta) * torch.sin(phi)
z = torch.cos(theta)

models_FM = {}
losses_FM = {}
metrics_FM = {}
for test in tests:
    print(test)
    generate_R_0, generate_R_1 = data_generator(test, ε=config[test]["epsilon"])

    R_0s = generate_R_0(32)
    R_1s = generate_R_1(32)

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.view_init(elev=15, azim=15)
    ax.plot_surface(x, y, z, color='cyan', alpha=0.25, edgecolor=None)
    plot_R(ax, R_0s, color="blue")
    plot_R(ax, R_1s, color="green")

    ax.set_xlim([-1, 1])
    ax.set_xticks([])
    ax.set_ylim([-1, 1])
    ax.set_yticks([])
    ax.set_zlim([-1, 1])
    ax.set_zticks([])
    ax.set_aspect("equal")

    models_FM[test], losses_FM[test], metrics_FM[test] = train_model(
        generate_R_0, generate_R_1, config[test], *eval_data[test]
    )

## Testing

In [ ]:
def create_animations(tests, models_FM, N_samples=2**5, N_steps=120):
    # Forward flow
    R_0s_test = {}
    R_ts_test = {}
    for test in tests:
        generate_R_0, _ = data_generator(test, ε=config[test]["epsilon"])
        R_0s_test[test] = generate_R_0(N_samples).to(device)
        R_ts_test[test] = R_0s_test[test].detach().clone()
        
        models_FM[test].eval()

    t = 0
    Δt = 1. / N_steps

    for test in tests:
        metadata = {'title': f'Flow Matching SO(3) {test}', 'artist': 'Matplotlib'}
        writer = FFMpegWriter(fps=30, metadata=metadata)

        fig = plt.figure(figsize=(8, 8))
        ax = fig.add_subplot(111, projection='3d')
        ax.view_init(elev=15, azim=15)
        ax.plot_surface(x, y, z, color='cyan', alpha=0.25, edgecolor=None)
        quiver = ax.quiver([], [], [], [], [], [])

        ax.set_xlim([-1, 1])
        ax.set_xticks([])
        ax.set_ylim([-1, 1])
        ax.set_yticks([])
        ax.set_zlim([-1, 1])
        ax.set_zticks([])
        ax.set_aspect("equal")
        with writer.saving(fig, f"output/flow_matching_SO3_{test}.mp4", dpi=150):

            quiver.remove()
            quiver = plot_R(ax, R_ts_test[test])
        
            writer.grab_frame()
            for frame in tqdm(range(N_steps)):
                t = frame * Δt
                with torch.no_grad():
                    R_ts_test[test] = models_FM[test].step(R_ts_test[test], torch.tensor([t], device=device)[..., None], Δt)

                quiver.remove()
                quiver = plot_R(ax, R_ts_test[test], color="blue")
                writer.grab_frame()
            writer.grab_frame()

In [ ]:
if generate_videos:
    create_animations(tests, models_FM, N_samples=2**5)

In [ ]:
t = 0
N_steps = config_shared["N_steps"]
Δt = 1. / N_steps
N_show = 5
N_skip = int(N_steps / (N_show-1))

N_models = len(tests)
N_samples = 32

R_0s_test = {}
R_1s_test = {}
R_ts_test = {}
for test in tests:
    generate_R_0, generate_R_1 = data_generator(test, ε=config[test]["epsilon"])
    R_0s_test[test] = generate_R_0(N_samples).to(device)
    R_1s_test[test] = generate_R_1(N_samples).to(device)
    R_ts_test[test] = R_0s_test[test].detach().clone()
    
    models_FM[test].eval()

fig = plt.figure(figsize=(4.8, 1.6 * N_models * 3/3.1))
gs = gridspec.GridSpec(N_models, 4, width_ratios=[1, 1, 1, 0.1], height_ratios=N_models * [1.], wspace=0.1, hspace=0.1)
cax = fig.add_subplot(gs[:, 3])

ax = []
for i in range(N_models):
    ax.append([])
    for j in range(3):
        a = fig.add_subplot(gs[i, j], projection="3d")
        a.view_init(elev=15, azim=15)
        a.plot_surface(x, y, z, color='cyan', alpha=0.25, edgecolor=None)

        a.set_xlim([-1, 1])
        a.set_xticks([])
        a.set_ylim([-1, 1])
        a.set_yticks([])
        a.set_zlim([-1, 1])
        a.set_zticks([])
        a.set_aspect("equal")
        ax[i].append(a)
        
ax[0][0].set_title(r"$\mathfrak{X}_0$")
ax[0][1].set_title(r"$\mathfrak{X}_t$")
ax[0][2].set_title(r"$\mathfrak{X}_1$")
Δc = 1 / (N_show - 1)
colors = [(j * Δc, 0.1, 1 - j * Δc) for j in range(N_show)]
cmap = mcolors.ListedColormap(colors)
for i, test in enumerate(tests):
    k = 0

    plot_R(ax[i][0], R_0s_test[test])
    plot_R(ax[i][2], R_1s_test[test])

    alpha = 1
    N_samples_shown = N_samples
    for j in range(N_steps+1):
        t = j * Δt
        if j == N_steps:
            alpha = 1
            N_samples_shown = N_samples
        with torch.no_grad():
            R_ts_test[test] = models_FM[test].step(R_ts_test[test], torch.tensor([t], device=device), torch.tensor([Δt], device=device))

        if j % N_skip == 0:

            plot_R(ax[i][1], R_ts_test[test], color=colors[k], alpha=alpha)
            k += 1
        alpha = 0.5
        N_samples_shown = N_samples // 4
fig.colorbar(ScalarMappable(cmap=cmap), cax=cax, ticks=np.linspace(0, 1, N_show), label="$t$");
fig.savefig("output/interpolation_SO3.pdf", bbox_inches="tight")

In [ ]:
N_EVAL_FINAL = 2**10

def print_metrics(name, metrics):
    print(f"{name:>10}: " + "  ".join(f"{key} = {value:.4f}" for key, value in metrics.items()))

eval_data_final = {}
for test in tests:
    generate_R_0, generate_R_1 = data_generator(test, ε=config[test]["epsilon"])
    R_0s_eval, R_1s_eval, R_1s_ref = generate_R_0(N_EVAL_FINAL), generate_R_1(N_EVAL_FINAL), generate_R_1(N_EVAL_FINAL)
    floor = compare(so3, R_1s_ref, R_1s_eval)
    eval_data_final[test] = (R_0s_eval, R_1s_eval, floor)

    print(test)
    print_metrics("floor", floor)
    print_metrics("model", models_FM[test].evaluate(R_0s_eval, R_1s_eval, N_steps=config[test]["N_steps"]))
    print_metrics("no model", compare(so3, R_0s_eval, R_1s_eval))

In [ ]:
steps = 2**np.arange(9)
keys = {"W_1": "$W_1$", "precision": "precision", "coverage": "coverage"}

fig, ax = plt.subplots(
    1, len(tests), figsize=(latex_line_width, 0.45 * latex_line_width), sharey=True, squeeze=False
)
ax = ax[0]
for a, test in zip(ax, tests):
    R_0s_eval, R_1s_eval, floor = eval_data_final[test]
    metrics = [models_FM[test].evaluate(R_0s_eval, R_1s_eval, N_steps=n) for n in tqdm(steps)]
    for (key, label), color in zip(keys.items(), ("tab:blue", "tab:orange", "tab:green")):
        a.plot(steps, [m[key] for m in metrics], marker=".", color=color, label=label)
        a.axhline(floor[key], color=color, linestyle="--", linewidth=0.8)
    a.set_xscale("log", base=2)
    a.set_yscale("log")
    a.set_xlabel("Integration steps")
    a.set_title(test.replace("_", " "))
ax[0].set_ylabel("Distance")
ax[0].legend();